# Vector Search y Embeddings (Introducción Práctica)

Objetivo: Introducir búsqueda vectorial (similaridad semántica) con embeddings sintéticos y cálculo de top-k vecinos usando distancia coseno. Se muestra pipeline básico y limitaciones frente a motores especializados (FAISS, Milvus, Qdrant).

---
## 1. Conceptos Clave
- **Embedding:** representación numérica d-dimensional de contenido (texto, imagen, entidad).
- **Vector DB:** almacén optimizado para búsquedas de similitud (ANN) en alta dimensionalidad.
- **Métrica de similitud:** coseno, Euclidiana, dot product.
- **ANN Index:** estructuras como HNSW, IVF, PQ para acelerar búsquedas.

Limitación local: usaremos NumPy y búsquedas exactas (O(n*d)).

---
## 2. Generación de Embeddings Sintéticos
Simulamos embeddings de productos y una consulta (query vector).

<!-- course-nav -->
> Ruta: [Inicio del repositorio](../README.md) | [Índice del nivel](README.md)  
> Anterior: [Caso Práctico: Streaming + Inferencia en Tiempo Real](19_streaming_inferencia_tiempo_real_caso_practico.ipynb) | Siguiente: [Vector Search: FAISS vs Búsqueda Exacta (Brute Force)](21_vector_search_faiss_vs_bruteforce.ipynb)


<!-- course-quality-structure -->
## Prerrequisitos

- Haber revisado los notebooks anteriores de la ruta o contar con experiencia equivalente.
- Conocer los conceptos básicos mencionados en el README del nivel.
- Tener disponible el repositorio local y las dependencias de `requirements.txt` cuando se ejecuten celdas de código.

## Preparación

1. Trabaja desde la raíz del repositorio.
2. Activa tu entorno de Python.
3. Instala dependencias con `pip install -r requirements.txt` si aún no lo hiciste.
4. Ejecuta las celdas en orden para conservar el contexto de variables y resultados.

## Mapa del notebook

1. 1. Conceptos Clave
2. 2. Generación de Embeddings Sintéticos
3. 3. Similitud Coseno y Búsqueda Top-k
4. 4. Filtrado y Segmentación Post-Búsqueda
5. 5. Búsqueda Exacta vs Aproximada
6. 6. Ejemplo de Re-Evaluación (Re-Ranking)
7. 7. Limitaciones y Próximos Pasos
8. 8. Ejercicio

## Diagrama de continuidad

```mermaid
flowchart LR
    A["Notebook anterior"] --> B["Vector Search y Embeddings (Introducción Práctica)"] --> C["Notebook siguiente"]
```


In [ ]:
import numpy as np, pandas as pd
np.random.seed(42)
DIM = 64  # dimensión simulada
N = 500   # número de elementos
embeddings = np.random.randn(N, DIM).astype('float32')
# Normalizamos para aproximar vectores unitarios (frecuente en coseno)
embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)
ids = [f'item_{i}' for i in range(N)]
df_meta = pd.DataFrame({'id': ids, 'category': np.random.choice(['A','B','C'], size=N)})
query = np.random.randn(DIM).astype('float32')
query /= np.linalg.norm(query)
query[:5]  # muestra parcial del vector

---
## 3. Similitud Coseno y Búsqueda Top-k
Formula coseno: cos(θ)= (a·b)/(||a||*||b||); para vectores normalizados equivale al dot product.

In [ ]:
# dot product directo (ya normalizado)
scores = embeddings @ query  # shape (N,)
k = 10
idx = np.argpartition(scores, -k)[-k:]  # índices top-k desordenados
top = idx[np.argsort(scores[idx])[::-1]]  # orden descendente
results = pd.DataFrame({'id': [ids[i] for i in top], 'score': scores[top]})
results

---
## 4. Filtrado y Segmentación Post-Búsqueda
Podemos aplicar filtros por metadatos (e.g., categoría).

In [ ]:
res_merged = results.merge(df_meta, on='id')
res_merged

---
## 5. Búsqueda Exacta vs Aproximada
| Aspecto | Exacta (brute force) | Aproximada (FAISS/Milvus) |
|---------|--------------------|---------------------------|
| Complejidad | O(n*d) | Sub-lineal (dependiendo del índice) |
| Precisión | 100% | ≥ ~90-99% según configuración |
| Latencia (grande) | Alta | Baja |
| Actualizaciones | Simples | Variable según estructura |
| Uso memoria | Embeddings + meta | Embeddings + estructuras índice |

---
## 6. Ejemplo de Re-Evaluación (Re-Ranking)
Generamos un segundo vector de contexto y combinamos scores.

In [ ]:
context = np.random.randn(DIM).astype('float32'); context /= np.linalg.norm(context)
context_scores = embeddings @ context
combined = 0.7*scores + 0.3*context_scores
idx2 = np.argpartition(combined, -k)[-k:]
top2 = idx2[np.argsort(combined[idx2])[::-1]]
reranked = pd.DataFrame({'id':[ids[i] for i in top2], 'combined_score': combined[top2]})
reranked

---
## 7. Limitaciones y Próximos Pasos
- Sin índices ANN: latencia escala lineal.
- Sin persistencia ni actualizaciones incrementales.
- No se manejan deletions/versionado.

**Extender:** probar FAISS (GPU/CPU), Milvus cloud-native, Qdrant con payload filters, evaluación de recall vs exacta, almacenar embeddings en formatos columnares.

---
## 8. Ejercicio
1. Ajusta DIM y N y mide tiempo de búsqueda.
2. Implementa métrica Euclidiana y compara orden.
3. Simula inserción de nuevos items y evalúa coste.

<!-- course-quality-close -->
## Ejercicio de cierre

Revisa el caso trabajado en el notebook y documenta una decisión de arquitectura: contexto, opción elegida, alternativas descartadas, riesgos y métricas de validación.

## Evaluación

- El notebook se ejecuta en orden sin depender de estado oculto.
- Las decisiones técnicas están justificadas con trade-offs claros.
- Los resultados, diagramas o tablas apoyan la explicación y no reemplazan el razonamiento.
- Las referencias usadas son oficiales, primarias o están incluidas en la bibliografía del curso.

## Referencias

- FAISS: https://github.com/facebookresearch/faiss
- Referencias del curso: ../04-recursos/referencias.md
